<a href="https://colab.research.google.com/github/budennovsk/Pandas/blob/master/scipy_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

from scipy.optimize import differential_evolution

# -----------------------
# 1) TRAIN DATA (как у тебя)
# -----------------------
data_train = [
    # Нержавейка 1 мм Азот S1.5
    dict(target=1, thickness=1.0, metal="Нержавейка", gas="Азот", nozzle="S1.5",
         speed=2.7, height=0.3, pressure=16.0, power_pct=100.0, power_w=3000.0, duty_pct=100.0, freq=5000.0,
         focus_cnc=-6.3, focus_corr=-8.3),
    dict(target=2, thickness=1.0, metal="Нержавейка", gas="Азот", nozzle="S1.5",
         speed=2.7, height=0.3, pressure=16.0, power_pct=100.0, power_w=3000.0, duty_pct=100.0, freq=5000.0,
         focus_cnc=-7.8, focus_corr=-8.3),
    dict(target=2, thickness=1.0, metal="Нержавейка", gas="Азот", nozzle="S1.5",
         speed=2.7, height=0.3, pressure=16.0, power_pct=100.0, power_w=3000.0, duty_pct=100.0, freq=5000.0,
         focus_cnc=-7.8, focus_corr=-8.8),
    dict(target=1, thickness=1.0, metal="Нержавейка", gas="Азот", nozzle="S1.5",
         speed=2.5, height=0.3, pressure=16.0, power_pct=100.0, power_w=3000.0, duty_pct=100.0, freq=5000.0,
         focus_cnc=-7.8, focus_corr=-8.8),
    dict(target=2, thickness=1.0, metal="Нержавейка", gas="Азот", nozzle="S1.5",
         speed=2.3, height=0.3, pressure=16.0, power_pct=100.0, power_w=3000.0, duty_pct=100.0, freq=5000.0,
         focus_cnc=-7.8, focus_corr=-8.8),
    dict(target=1, thickness=1.0, metal="Нержавейка", gas="Азот", nozzle="S1.5",
         speed=2.3, height=0.2, pressure=16.0, power_pct=100.0, power_w=3000.0, duty_pct=100.0, freq=5000.0,
         focus_cnc=-7.8, focus_corr=-8.8),
    dict(target=2, thickness=1.0, metal="Нержавейка", gas="Азот", nozzle="S1.5",
         speed=2.3, height=0.2, pressure=15.0, power_pct=100.0, power_w=3000.0, duty_pct=100.0, freq=5000.0,
         focus_cnc=-7.8, focus_corr=-8.8),
    dict(target=3, thickness=1.0, metal="Нержавейка", gas="Азот", nozzle="S1.5",
         speed=2.3, height=0.2, pressure=14.0, power_pct=100.0, power_w=3000.0, duty_pct=100.0, freq=5000.0,
         focus_cnc=-7.8, focus_corr=-8.8),
    dict(target=1, thickness=1.0, metal="Нержавейка", gas="Азот", nozzle="S1.5",
         speed=2.3, height=0.2, pressure=13.0, power_pct=100.0, power_w=3000.0, duty_pct=100.0, freq=5000.0,
         focus_cnc=-7.8, focus_corr=-8.8),
    dict(target=0, thickness=1.0, metal="Нержавейка", gas="Азот", nozzle="S1.5",
         speed=2.3, height=0.2, pressure=14.0, power_pct=100.0, power_w=3000.0, duty_pct=100.0, freq=5000.0,
         focus_cnc=-7.8, focus_corr=-8.8),

    # Сталь3 12 мм Кислород S2.5
    dict(target=0, thickness=12.0, metal="Сталь3", gas="Кислород", nozzle="S2.5",
         speed=2.1, height=0.2, pressure=0.6, power_pct=100.0, power_w=3000.0, duty_pct=100.0, freq=5000.0,
         focus_cnc=-4.0, focus_corr=-6.0),

    # # Алюминий 2 мм Воздух S2
    # dict(target=1, thickness=2.0, metal="Алюминий", gas="Воздух", nozzle="S2",
    #      speed=18.0, height=0.2, pressure=0.6, power_pct=100.0, power_w=3000.0, duty_pct=100.0, freq=5000.0,
    #      focus_cnc=-4.5, focus_corr=-6.0),
    # dict(target=3, thickness=2.0, metal="Алюминий", gas="Воздух", nozzle="S2",
    #      speed=18.0, height=0.2, pressure=0.6, power_pct=100.0, power_w=3000.0, duty_pct=100.0, freq=5000.0,
    #      focus_cnc=-5.0, focus_corr=-6.0),
    # dict(target=3, thickness=2.0, metal="Алюминий", gas="Воздух", nozzle="S2",
    #      speed=18.0, height=0.2, pressure=0.6, power_pct=100.0, power_w=3000.0, duty_pct=100.0, freq=5000.0,
    #      focus_cnc=-4.0, focus_corr=-6.0),
    # dict(target=0, thickness=2.0, metal="Алюминий", gas="Воздух", nozzle="S2",
    #      speed=18.0, height=0.2, pressure=0.6, power_pct=100.0, power_w=3000.0, duty_pct=100.0, freq=5000.0,
    #      focus_cnc=-3.5, focus_corr=-6.0),
    # dict(target=3, thickness=2.0, metal="Алюминий", gas="Воздух", nozzle="S2",
    #      speed=18.0, height=0.2, pressure=0.6, power_pct=100.0, power_w=3000.0, duty_pct=100.0, freq=5000.0,
    #      focus_cnc=-1.5, focus_corr=-6.0),
    # dict(target=0, thickness=2.0, metal="Алюминий", gas="Воздух", nozzle="S2",
    #      speed=17.0, height=0.2, pressure=0.6, power_pct=100.0, power_w=3000.0, duty_pct=100.0, freq=5000.0,
    #      focus_cnc=-1.5, focus_corr=-6.0),
]

df_train = pd.DataFrame(data_train)

# -----------------------
# 2) TEST DATA (ты подаёшь одну итерацию)
#    target здесь НЕ нужен для рекомендации, но можно хранить для контроля.
# -----------------------
data_test = [
    {"id": 11, "target": 1, "thickness": 2.0, "metal": "Алюминий", "gas": "Воздух", "nozzle": "S2",
     "speed": 18.0, "height": 0.2, "pressure": 0.6, "power_pct": 100.0, "power_w": 3000.0,
     "duty_pct": 100.0, "freq": 5000.0, "focus_cnc": -4.5, "focus_corr": -6.0}
]
df_test = pd.DataFrame(data_test)

# -----------------------
# 3) Обучаем модель на TRAIN
# -----------------------
cat_cols = ["metal", "gas", "nozzle"]
num_cols = ["thickness", "speed", "height", "pressure", "power_pct", "power_w", "duty_pct", "freq", "focus_cnc", "focus_corr"]
feature_cols = cat_cols + num_cols

X_train = df_train[feature_cols]
y_train = df_train["target"].astype(int)

pre = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
     ("num", "passthrough", num_cols)]
)

clf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight="balanced",
)

pipe = Pipeline([("pre", pre), ("clf", clf)])
pipe.fit(X_train, y_train)

classes = pipe.named_steps["clf"].classes_
idx0 = int(np.where(classes == 0)[0][0])

# -----------------------
# 4) Функция: рекомендовать новый режим для каждой строки df_test
#    Мы фиксируем категориальные признаки + толщину и оптимизируем числовые параметры.
# -----------------------
def recommend_for_row(row, pad=0.3):
    context = {
        "metal": row["metal"],
        "gas": row["gas"],
        "nozzle": row["nozzle"],
        "thickness": float(row["thickness"]),
    }
    fixed = {
        "power_pct": float(row["power_pct"]),
        "power_w": float(row["power_w"]),
        "duty_pct": float(row["duty_pct"]),
        "freq": float(row["freq"]),
    }

    # Берём под-датасет по такому же контексту, чтобы задать границы поиска реалистично
    mask = (
        (df_train["metal"] == context["metal"]) &
        (df_train["gas"] == context["gas"]) &
        (df_train["nozzle"] == context["nozzle"]) &
        (df_train["thickness"] == context["thickness"])
    )
    df_ctx = df_train[mask].copy()

    # Если в train нет точек для такого контекста — fallback: используем глобальные границы
    if len(df_ctx) == 0:
        df_ctx = df_train.copy()

    def bounds_from(col, default, pad_):
        lo = float(df_ctx[col].min()) - pad_
        hi = float(df_ctx[col].max()) + pad_
        if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
            lo, hi = default
        return (lo, hi)

    # bounds (подбираем только эти параметры)
    bounds = [
        bounds_from("speed", (0.1, 30.0),  pad),
        bounds_from("height", (0.05, 1.0), pad/3),
        bounds_from("pressure", (0.1, 25.0), pad*2),
        bounds_from("focus_cnc", (-30.0, 30.0), pad),
        bounds_from("focus_corr", (-30.0, 30.0), pad),
    ]

    # квантование (под твой станок можно поменять)
    def quantize(x):
        speed, height, pressure, focus_cnc, focus_corr = x
        speed = np.round(speed / 0.1) * 0.1
        height = np.round(height / 0.1) * 0.1
        pressure = np.round(pressure / 0.1) * 0.1
        focus_cnc = np.round(focus_cnc / 0.5) * 0.5
        focus_corr = np.round(focus_corr / 0.5) * 0.5
        return np.array([speed, height, pressure, focus_cnc, focus_corr], dtype=float)

    def objective(x):
        xq = quantize(x)
        feat = dict(context)
        feat.update(fixed)
        feat.update({
            "speed": float(xq[0]),
            "height": float(xq[1]),
            "pressure": float(xq[2]),
            "focus_cnc": float(xq[3]),
            "focus_corr": float(xq[4]),
        })
        proba = pipe.predict_proba(pd.DataFrame([feat])[feature_cols])[0]
        p0 = float(proba[idx0])
        return -p0  # хотим максимиз��ровать p0

    res = differential_evolution(
        objective,
        bounds=bounds,
        seed=42,
        popsize=10,
        maxiter=50,
        tol=1e-3,
        polish=False
    )

    x_best = quantize(res.x)

    best_feat = dict(context)
    best_feat.update(fixed)
    best_feat.update({
        "speed": float(x_best[0]),
        "height": float(x_best[1]),
        "pressure": float(x_best[2]),
        "focus_cnc": float(x_best[3]),
        "focus_corr": float(x_best[4]),
    })

    p0_best = float(pipe.predict_proba(pd.DataFrame([best_feat])[feature_cols])[0][idx0])

    return best_feat, p0_best, df_ctx

# -----------------------
# 5) Применяем к df_test
# -----------------------
for i, row in df_test.iterrows():
    rec, p0, df_ctx = recommend_for_row(row)

    print("\n=== Вход (текущая итерация оператора) ===")
    print(row.to_dict())

    print("\n=== Контекст в train (строк найдено):", len(df_ctx), "===")

    print("\n=== Рекомендация следующего режима ===")
    print(rec)
    print(f"P(target=0) ~ {p0:.3f}")

In [ ]:
import pandas as pd

data_1 = [
    {"id": 11, "target": 1, "thickness": 2.0, "metal": "Алюминий", "gas": "Воздух", "nozzle": "S2",
     "speed": 18.0, "height": 0.2, "pressure": 0.6, "power_pct": 100.0, "power_w": 3000.0,
     "duty_pct": 100.0, "freq": 5000.0, "focus_cnc": -4.5, "focus_corr": -6.0},

    {"id": 12, "target": 2, "thickness": 2.0, "metal": "Алюминий", "gas": "Воздух", "nozzle": "S2",
     "speed": 18.0, "height": 0.2, "pressure": 0.6, "power_pct": 100.0, "power_w": 3000.0,
     "duty_pct": 100.0, "freq": 5000.0, "focus_cnc": -5.0, "focus_corr": -6.0},

    {"id": 13, "target": 2, "thickness": 2.0, "metal": "Алюминий", "gas": "Воздух", "nozzle": "S2",
     "speed": 18.0, "height": 0.2, "pressure": 0.6, "power_pct": 100.0, "power_w": 3000.0,
     "duty_pct": 100.0, "freq": 5000.0, "focus_cnc": -4.0, "focus_corr": -6.0},

    {"id": 14, "target": 1, "thickness": 2.0, "metal": "Алюминий", "gas": "Воздух", "nozzle": "S2",
     "speed": 18.0, "height": 0.2, "pressure": 0.6, "power_pct": 100.0, "power_w": 3000.0,
     "duty_pct": 100.0, "freq": 5000.0, "focus_cnc": -3.5, "focus_corr": -6.0},

    {"id": 15, "target": 2, "thickness": 2.0, "metal": "Алюминий", "gas": "Воздух", "nozzle": "S2",
     "speed": 18.0, "height": 0.2, "pressure": 0.6, "power_pct": 100.0, "power_w": 3000.0,
     "duty_pct": 100.0, "freq": 5000.0, "focus_cnc": -1.5, "focus_corr": -6.0},

    {"id": 16, "target": 0, "thickness": 2.0, "metal": "Алюминий", "gas": "Воздух", "nozzle": "S2",
     "speed": 17.0, "height": 0.2, "pressure": 0.6, "power_pct": 100.0, "power_w": 3000.0,
     "duty_pct": 100.0, "freq": 5000.0, "focus_cnc": -1.5, "focus_corr": -6.0},
]

df_1 = pd.DataFrame(data_1)

# чтобы столбцы были в "красивом" порядке
df_1 = df_1[[
    "id", "target", "thickness", "metal", "gas", "nozzle",
    "speed", "height", "pressure",
    "power_pct", "power_w", "duty_pct", "freq",
    "focus_cnc", "focus_corr"
]]

df_1